# [P2] Q1 B + D: analysis and realizations

**Owner:** Part 2  
**Temporary part notebook.** At integration it is merged into `Q1_neural_spikes.ipynb` and this file is deleted.


## Before writing any code

Tasks: `t_q1b` (analysis and comparison), `t_q1d` (new realizations, how to tell the patterns apart).

Decisions to read on the team board first (board ids): `b_*` (B), `d_n d_props d_plot d_cls d_dur` (D), and the shared `a_rate`.

Do not write code for a task until its decisions are made. Mark your cells with the part tag so the merge is easy.


## Q1 B, step 1: the statistics functions

Decisions applied: `b_stats` (rate and mean interval, CV, Fano, LV, short-interval fraction, serial correlation), `b_thr` (a curve over thresholds), `b_fano` (a curve over window sizes) and `b_bin` (100 ms bins).

Each function takes the spike times in seconds and returns a number, or `np.nan` when the number is undefined (too few spikes, division by zero). Nothing here depends on the simulation yet: the functions are tested on inputs whose answer we know in advance (decision `g_verify`).

In [ ]:
import numpy as np
from q1_stats import (isi, mean_rate, mean_isi, cv, lv, next_interval_corr, short_isi_fraction, short_isi_curve,
                      window_counts, fano, fano_curve, rate_over_time)

# HE: בדיקה 1: מקרים שאת התשובה שלהם אפשר לחשב ביד.
# HE: פעימות ב-0, 1, 4, 5 שניות => מרווחים 1, 3, 1.
s = np.array([0.0, 1.0, 4.0, 5.0])
assert np.allclose(isi(s), [1, 3, 1])
assert mean_rate(s, T=10) == 0.4                      # HE: 4 פעימות ב-10 שניות
assert np.isclose(mean_isi(s), 5 / 3)

# HE: שני מרווחים בלבד, 1 ו-3: ממוצע 2, סטיית תקן (ddof=1) היא שורש 2, ולכן CV = שורש 2 חלקי 2.
two = np.array([0.0, 1.0, 4.0])
assert np.isclose(cv(two), np.sqrt(2) / 2)
# HE: LV = 3 * ((1-3)/(1+3))^2 = 3 * 0.25 = 0.75.
assert np.isclose(lv(two), 0.75)

# HE: מרווחים מתחלפים 1,2,1,2,1: כל קצר בא אחרי ארוך, ולכן המתאם הסדרתי הוא בדיוק -1.
alt = np.concatenate([[0.0], np.cumsum([1, 2, 1, 2, 1])])
assert np.isclose(next_interval_corr(alt), -1.0)
print("hand-computed cases passed")

In [ ]:
# HE: בדיקה 2: נוירון רגולרי לגמרי (פעימה כל 100 מילי-שניות) הוא "אפס שונות": CV, LV ו-Fano שווים 0.
regular = np.arange(0, 10, 0.1)
assert np.isclose(cv(regular), 0, atol=1e-9) and np.isclose(lv(regular), 0, atol=1e-9)
assert np.isclose(fano(regular, T=10, window=0.5), 0, atol=1e-9)   # HE: בכל חלון בדיוק 5 פעימות
assert np.isnan(next_interval_corr(regular))                             # HE: אין שונות, אין מתאם: nan
assert short_isi_fraction(regular, 0.010) == 0                          # HE: אין מרווח קצר מ-10 מילי-שניות

# HE: בדיקה 3: נוירון אקראי (פואסוני) צריך לתת בערך 1 בשלושת המדדים CV, LV ו-Fano. מחוללים אותו כאן עם זרע קבוע.
rng = np.random.default_rng(1)
poisson = np.cumsum(rng.exponential(1 / 10, size=20000))            # HE: קצב 10Hz, כ-2000 שניות
T_long = poisson[-1]
poisson = poisson[:-1]
for name, value in [("CV", cv(poisson)), ("LV", lv(poisson)),
                    ("Fano(500ms)", fano(poisson, T_long, 0.5))]:
    print(f"{name:12s}", round(value, 3))
    assert abs(value - 1) < 0.1, name
# HE: במרווחים אקראיים אין זיכרון, לכן המתאם הסדרתי קרוב ל-0.
assert abs(next_interval_corr(poisson)) < 0.05
# HE: במרווחים מעריכיים, שיעור המרווחים הקצרים מ-10ms הוא 1-exp(-10Hz*0.01s) = 0.0952.
print("short-interval fraction:", round(short_isi_fraction(poisson, 0.010), 4), "expected", round(1 - np.exp(-0.1), 4))
assert abs(short_isi_fraction(poisson, 0.010) - (1 - np.exp(-0.1))) < 0.01
print("regular and random-neuron cases passed")

In [ ]:
# HE: בדיקה 4: מקרי קצה. כל מדד שאי אפשר לחשב מחזיר nan, ולא שגיאה (כלל מההוראות).
empty = np.array([])
one = np.array([0.5])
three = np.array([0.1, 0.2, 0.35])
for f in (mean_isi, cv, lv, next_interval_corr, short_isi_fraction):
    assert np.isnan(f(empty)) and np.isnan(f(one)), f.__name__
assert np.isnan(cv(one)) and np.isnan(lv(three[:2])) and not np.isnan(lv(three))
assert np.isnan(fano(empty, T=5, window=0.5))                       # HE: ממוצע ספירות 0 => nan
assert np.isnan(fano(three, T=0.4, window=0.5))                     # HE: פחות משני חלונות => nan
assert mean_rate(empty, T=5) == 0

# HE: חלונות: הקלטה של 5 שניות וחלון של 0.5 נותנים בדיוק 10 חלונות, וסכום הספירות שווה למספר הפעימות.
c = window_counts(poisson[poisson < 5], T=5, window=0.5)
assert len(c) == 10 and c.sum() == int((poisson < 5).sum())
# HE: קצב לאורך זמן בחלונות של 100ms: 50 חלונות, וממוצע הקצב שווה לקצב הכולל.
r = rate_over_time(poisson[poisson < 5], T=5, bin_size=0.1)
assert len(r) == 50 and np.isclose(r.mean(), len(poisson[poisson < 5]) / 5)
# HE: העקומות מחזירות ערך לכל סף/חלון, באורך שווה לרשימה שהוזנה.
assert len(short_isi_curve(poisson, np.linspace(0.002, 0.03, 8))) == 8
assert len(fano_curve(poisson, T_long, [0.05, 0.1, 0.5])) == 3
print("edge cases passed: undefined values return nan, never an exception")

**What step 1 shows**

- The functions agree with answers computed by hand (for example, intervals 1 and 3 give CV = 0.707, LV = 0.75).
- A perfectly regular neuron gives CV, LV and Fano equal to 0, and an undefined serial correlation (`nan`).
- A random (Poisson) neuron gives CV, LV and Fano close to 1 (0.998, 1.001, 0.996) and a serial correlation near 0. The short-interval fraction below 10 ms is 0.098, close to the theoretical 0.095.
- Undefined cases (no spikes, one spike, too few windows) return `nan`, never an exception.

## Q1 B, step 2: raster and ISI histograms

Part 1's simulation (`simulate_regular`, `simulate_irregular`, `simulate_bursting` in `q1_models.py`) is now available, so every cell below runs on it. The cell marked `STAND-IN` is only a fallback that was used while Part 1 had not delivered: it switches on automatically if `q1_models.py` loses one of the four contract functions (`make_rngs` and the three `simulate_*`), and it will be deleted at integration. Figures are saved to `figures/` as 200 dpi PNG files (decision `g_figs`) only when the data come from Part 1's simulation. All three neurons have the same mean rate of 10 Hz (decision `a_rate`).

In [ ]:
# STAND-IN (temporary): used only while Part 1's simulation is not in q1_models.py.
# HE: מחוללים זמניים כדי לבנות גרפים. לא המודלים האמיתיים של שחר. כולם עם קצב ממוצע של 10Hz.
# HE: המתג: אם ב-q1_models כבר יש את ארבע הפונקציות של החוזה, המחברת משתמשת בהן אוטומטית, וכל המחברת רצה עליהן.
import os
import numpy as np
import matplotlib.pyplot as plt
import q1_models

PART1_FUNCS = ("make_rngs", "simulate_regular", "simulate_irregular", "simulate_bursting")
USE_STANDIN = not all(hasattr(q1_models, f) for f in PART1_FUNCS)
print("data source:", "STAND-IN neurons (Part 1 not delivered yet)" if USE_STANDIN else "Part 1's simulation (q1_models)")

# HE: שמירת גרפים (החלטה g_figs): PNG ב-200dpi לתיקייה figures. שומרים רק כשהנתונים אמיתיים, אף פעם לא את נתוני הדמה.
FIG_DIR = os.path.join("..", "figures")    # HE: מתוך parts. באיחוד, כשהמחברת בתיקייה הראשית, זה יהיה "figures".


def save_fig(fig, name):
    if USE_STANDIN:
        return
    os.makedirs(FIG_DIR, exist_ok=True)
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=200, bbox_inches="tight")

RATE = 10.0            # HE: קצב ממוצע בהרץ (החלטה a_rate)
NAMES = ["Regular", "Irregular (random)", "Bursting"]
COLORS = ["tab:blue", "tab:orange", "tab:green"]


def _to_spike_times(intervals, T):
    # HE: צוברים מרווחים לזמני פעימות, וגוזרים כך שהכול בתוך [0, T).
    t = np.cumsum(intervals)
    return t[t < T]


def standin_regular(T, rng):
    # HE: מרווחים סביב 100ms עם סטייה קטנה (התפלגות גמא עם צורה גדולה = כמעט קבוע).
    k = 25
    return _to_spike_times(rng.gamma(k, 1 / (RATE * k), size=int(RATE * T * 2) + 20), T)


def standin_irregular(T, rng):
    # HE: תהליך פואסוני: מרווחים מעריכיים.
    return _to_spike_times(rng.exponential(1 / RATE, size=int(RATE * T * 2) + 20), T)


def standin_bursting(T, rng, per_burst=5, within=0.005):
    # HE: פרצים: כל פרץ הוא 5 פעימות במרווחים של כ-5ms, ופרצים מתחילים באקראי בקצב RATE/5.
    starts = _to_spike_times(rng.exponential(per_burst / RATE, size=int(RATE * T) + 20), T)
    # HE: בהקלטה קצרה ייתכן שלא התחיל אף פרץ. אז מחזירים מערך ריק (במקום לקרוס).
    if len(starts) == 0:
        return np.array([])
    spikes = np.concatenate([s + np.concatenate([[0], np.cumsum(rng.exponential(within, per_burst - 1))])
                             for s in starts])
    spikes = np.sort(spikes[spikes < T])
    return spikes


def make_trains(T, seed):
    # HE: מחזירה שלושה רצפי פעימות (רגולרי, אקראי, פרצי) באורך T עם זרע נתון.
    # HE: עם הסימולציה של שחר: make_rngs נותנת שלושה מחוללים בלתי תלויים מאותו זרע, אחד לכל נוירון (לפי החוזה).
    if not USE_STANDIN:
        rngs = q1_models.make_rngs(seed, 3)
        return [q1_models.simulate_regular(T, rngs[0], rate_hz=RATE),
                q1_models.simulate_irregular(T, rngs[1], rate_hz=RATE),
                q1_models.simulate_bursting(T, rngs[2], rate_hz=RATE)]
    rng = np.random.default_rng(seed)
    return [standin_regular(T, rng), standin_irregular(T, rng), standin_bursting(T, rng)]


# HE: ריצה קצרה (5 שניות) לרסטר, וריצה ארוכה (200 שניות) כדי שההיסטוגרמות יהיו יציבות.
T_SHORT, T_LONG = 5.0, 200.0
short = make_trains(T_SHORT, seed=11)
long_ = make_trains(T_LONG, seed=12)

# HE: בדיקה: הקצב הממוצע של שלושת המחוללים קרוב ל-10Hz, והזמנים עולים בקפדנות ובתוך ההקלטה.
for name, s, T in zip(NAMES, long_, [T_LONG] * 3):
    assert np.all(np.diff(s) > 0) and s[0] >= 0 and s[-1] < T, name
    print(f"{name:20s} mean rate {len(s) / T:5.2f} Hz")
    assert abs(len(s) / T - RATE) < 1.0, name

In [ ]:
# HE: גרף 1: רסטר (החלטה b_figs). כל פעימה היא קו אנכי קטן, שורה לכל נוירון, ציר x הוא הזמן.
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.eventplot(short, colors=COLORS, lineoffsets=[2, 1, 0], linelengths=0.7)
ax.set_yticks([2, 1, 0]); ax.set_yticklabels(NAMES)
ax.set_xlim(0, T_SHORT); ax.set_xlabel("Time (s)")
ax.set_title("Raster: same mean rate (10 Hz), different timing structure")
plt.tight_layout(); save_fig(fig, "q1b_raster"); plt.show()

In [ ]:
# HE: גרף 2: היסטוגרמת מרווחים (ISI) בציר לוגריתמי (החלטה b_axis). ב-bins בלוגריתם, רוחב הפח גדל עם הערך.
from q1_stats import isi
bins = np.logspace(-3, 0, 37)              # HE: מ-1ms עד שנייה אחת (החלטה b_axis)
fig, axes = plt.subplots(1, 3, figsize=(11, 3), sharex=True, sharey=True)
for ax, name, color, s in zip(axes, NAMES, COLORS, long_):
    ax.hist(isi(s), bins=bins, color=color)
    ax.set_xscale("log"); ax.set_title(name); ax.set_xlabel("Interval (s, log axis)")
axes[0].set_ylabel("Count")
plt.tight_layout(); save_fig(fig, "q1b_isi_histograms"); plt.show()
# HE: מרווחים מחוץ לטווח 1ms עד 1s לא נכנסים לגרף. סופרים כמה כאלה יש, כדי לדווח בכנות מה לא מוצג.
for name, s in zip(NAMES, long_):
    x = isi(s)
    print(f"{name:20s} intervals outside 1 ms to 1 s: {int(((x < 1e-3) | (x > 1)).sum())} of {len(x)}")

**What step 2 shows (on Part 1's models)**

- **Raster:** all three rows have about 10 spikes per second, yet they look different. The regular neuron is evenly spaced. The random one has clusters and gaps by chance. The bursting one fires tight groups of spikes separated by quiet stretches. Mean rate alone does not tell them apart.
- **ISI histograms (log axis):** the regular neuron has one narrow peak near 100 ms. The random neuron has one wide hump from a few milliseconds up to about a second. The bursting neuron has two separate groups: very short intervals inside bursts (around 5 ms) and long ones between bursts (a few hundred ms). On a linear axis the short group would be squeezed against zero.
- **What the log axis leaves out:** the bins cover 1 ms to 1 s. Only 3 of the 2,053 bursting intervals fall outside, and none of the regular or random ones, so the figure shows almost everything.

## Q1 B, step 3: rate over time, return map, threshold curve and Fano curve

Decisions applied: `b_bin` (100 ms bins), `b_figs` (rate over time, return map), `b_thr` (a curve over thresholds from 2 to 30 ms) and `b_fano` (a curve over window sizes). On Part 1's models.

In [ ]:
from q1_stats import rate_over_time, short_isi_curve, fano_curve

# HE: גרף 3: קצב לאורך זמן בחלונות של 100ms (החלטה b_bin). חלון קצר מראה פרצים אבל רועש: זה בדיוק הלקח.
fig, ax = plt.subplots(figsize=(10, 3))
t_bins = np.arange(int(T_SHORT / 0.1)) * 0.1 + 0.05
for name, color, s in zip(NAMES, COLORS, short):
    ax.plot(t_bins, rate_over_time(s, T_SHORT, 0.100), color=color, label=name, lw=1.5)
ax.set_xlabel("Time (s)"); ax.set_ylabel("Rate (Hz, 100 ms bins)"); ax.legend(ncol=3, loc="upper right")
ax.set_title("Rate in 100 ms bins: the same mean, very different fluctuations")
plt.tight_layout(); save_fig(fig, "q1b_rate_100ms"); plt.show()

In [ ]:
# HE: גרף 4: מפת חזרה. כל נקודה היא מרווח (ציר x) מול המרווח שאחריו (ציר y), בצירים לוגריתמיים.
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), sharex=True, sharey=True)
for ax, name, color, s in zip(axes, NAMES, COLORS, long_):
    x = isi(s)
    ax.scatter(x[:-1], x[1:], s=3, alpha=0.3, color=color)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_title(name); ax.set_xlabel("Interval n (s)")
    # HE: מגבילים את הצירים לטווח הרלוונטי (1ms עד 3 שניות), כדי שחריגים נדירים לא ימתחו את הגרף.
    ax.set_xlim(1e-3, 3); ax.set_ylim(1e-3, 3)
axes[0].set_ylabel("Interval n+1 (s)")
plt.tight_layout(); save_fig(fig, "q1b_return_map"); plt.show()

In [ ]:
# HE: גרף 5: עקומת סף (החלטה b_thr): שיעור המרווחים הקצרים מהסף, לכל סף בין 2 ל-30 מילי-שניות.
thresholds = np.linspace(0.002, 0.030, 15)
fig, ax = plt.subplots(figsize=(6, 3.4))
for name, color, s in zip(NAMES, COLORS, long_):
    ax.plot(thresholds * 1000, short_isi_curve(s, thresholds), color=color, marker="o", ms=3, label=name)
ax.set_xlabel("Short-interval threshold (ms)"); ax.set_ylabel("Fraction of intervals below threshold")
ax.legend(); ax.set_title("How the answer depends on the threshold")
plt.tight_layout(); save_fig(fig, "q1b_threshold_curve"); plt.show()

# HE: בדיקות סבירות על הנתונים: ברגולרי אין מרווחים קצרים מ-10ms, בפרצי רוב המרווחים קצרים, באקראי כ-1-exp(-0.1)=0.095.
at10 = {n: short_isi_curve(s, [0.010])[0] for n, s in zip(NAMES, long_)}
print({k: round(v, 3) for k, v in at10.items()})
assert at10["Regular"] == 0 and at10["Bursting"] > 0.5 and abs(at10["Irregular (random)"] - 0.095) < 0.02

In [ ]:
# HE: גרף 6: עקומת Fano (החלטה b_fano): Fano לכל גודל חלון, בציר x לוגריתמי. הקו האפור ב-1 הוא נוירון אקראי.
windows = np.logspace(np.log10(0.05), 0, 14)  # HE: מ-50ms עד שנייה אחת (החלטה b_fano)
fig, ax = plt.subplots(figsize=(6, 3.4))
curves = {}
for name, color, s in zip(NAMES, COLORS, long_):
    curves[name] = fano_curve(s, T_LONG, windows)
    ax.plot(windows, curves[name], color=color, marker="o", ms=3, label=name)
ax.axhline(1, color="gray", ls="--", lw=1)
ax.set_xscale("log"); ax.set_xlabel("Window size (s)"); ax.set_ylabel("Fano factor"); ax.legend()
ax.set_title("Fano factor depends on the time scale")
plt.tight_layout(); save_fig(fig, "q1b_fano_curve"); plt.show()

# HE: בדיקות סבירות: רגולרי נמוך מ-1, אקראי קרוב ל-1, ובפרצי גבוה בחלונות קצרים ויורד עם החלון.
i1 = int(np.argmin(np.abs(windows - 1.0)))
print({n: round(float(c[i1]), 2) for n, c in curves.items()}, "at window", round(windows[i1], 2), "s")
assert curves["Regular"][i1] < 0.3
assert abs(curves["Irregular (random)"][i1] - 1) < 0.25
# HE: אצל שחר לפרצים יש שקט מינימלי, ולכן הם מגיעים בקצב כמעט קבוע. בחלון קצר Fano גבוה, ובחלון ארוך הוא יורד אבל נשאר מעל 1.
assert np.all(curves["Bursting"][windows <= 0.25] > 2) and np.all(curves["Bursting"] > 1)

**What step 3 shows (on Part 1's models)**

- **Rate in 100 ms bins:** the regular neuron stays flat near 10 Hz. The random one fluctuates between 0 and about 40 Hz. The bursting one jumps between 0 and 40 to 60 Hz. One 100 ms bin is a very noisy estimate of the rate, so a short-window rate is not a reliable measure.
- **Return map:** the regular neuron is one tight cloud (every interval is followed by one of about the same length). The random neuron is one wide cloud with no pattern. The bursting neuron has three clouds: short followed by short (inside a burst), short followed by long (the end of a burst) and long followed by short (the start of the next one). There is no long followed by long. The histogram cannot show this order.
- **Threshold curve:** the regular neuron stays at 0 for every threshold. The bursting neuron rises from 0.13 at 4 ms to 0.65 at 6 ms and reaches a plateau of 0.75 from 8 ms on, so its short intervals are all very short. The random neuron rises slowly and steadily (0.056 at 8 ms, 0.26 at 30 ms), because a random neuron has short intervals by chance. Because the bursting curve is flat from 8 ms, the conclusion does not depend on the exact threshold. The final threshold is **8 ms** (the student's decision): the first point where the bursting neuron is on its plateau while the random neuron is still low.
- **Fano curve (windows 50 to 1000 ms, decision `b_fano`):** the regular neuron starts at 0.5 and falls to about 0.01 as the window grows. The random neuron stays near 1 (0.95 to 1.14). The bursting neuron starts at about 3.3 in a 50 ms window and falls to 1.2 in a 1 s window. It falls because this model has a minimum quiet period between bursts, so bursts arrive at a nearly regular pace and a long window holds a similar number of bursts each time. So the Fano factor of a bursting neuron depends on the window, and a single window would hide that.

## Q1 B, step 4: one comparison figure and the numbers table

Decisions applied: `b_figs` (one comparison figure, the three neurons on the same axes and in the same colours; a small table of numbers next to the figures) and `b_stats` (rate with mean and standard deviation of the intervals, CV, Fano, LV, short-interval fraction, lag-1 serial correlation). On Part 1's models.

The diagram of how each neuron is built (also in `b_figs`) comes in step 5, because it describes Part 1's models.

In [ ]:
# HE: גרף השוואה אחד: שלוש התפלגויות המרווחים על אותם צירים ובאותם צבעים.
# HE: ציר y הוא החלק מכלל המרווחים שנופל בכל פח (משקל 1/N לכל מרווח), כדי שמספר המרווחים לא ישפיע.
# HE: לא משתמשים ב-density=True: עם פחים ברוחב משתנה (ציר לוגריתמי) היא מנפחת את הפחים הצרים של המרווחים הקצרים.
fig, ax = plt.subplots(figsize=(7, 3.6))
for name, color, s in zip(NAMES, COLORS, long_):
    x = isi(s)
    ax.hist(x, bins=bins, weights=np.ones(len(x)) / len(x), histtype="step", lw=2, color=color, label=name)
ax.set_xscale("log"); ax.set_xlabel("Interval (s, log axis)"); ax.set_ylabel("Fraction of intervals")
ax.legend(); ax.set_title("Same mean rate, three different interval distributions")
plt.tight_layout(); save_fig(fig, "q1b_isi_comparison"); plt.show()

In [ ]:
import pandas as pd
from q1_stats import mean_rate, mean_isi, cv, lv, fano, short_isi_fraction, next_interval_corr

FANO_WINDOW = 0.5     # HE: לטבלה צריך ערך אחד. העקומה (שלב 3) היא העיקר; כאן מדווחים את 500ms ומציינים זאת.
SHORT_THR = 0.008     # HE: הסף הסופי (החלטה b_thr): 8ms, הסף הראשון שבו הנוירון הפרצי כבר במישור (75%) והאקראי עדיין נמוך (5.6%).


def stats_table(spike_trains, T):
    # HE: טבלה: שורה לכל נוירון, עמודה לכל מדד מהחלטה b_stats.
    rows = {}
    for name, s in zip(NAMES, spike_trains):
        x = isi(s)
        rows[name] = {
            "rate (Hz)": mean_rate(s, T),
            "mean ISI (ms)": 1000 * mean_isi(s),
            "std ISI (ms)": 1000 * x.std(ddof=1) if len(x) >= 2 else np.nan,
            "CV": cv(s),
            "LV": lv(s),
            f"Fano ({int(FANO_WINDOW * 1000)} ms)": fano(s, T, FANO_WINDOW),
            f"short ISI < {int(SHORT_THR * 1000)} ms": short_isi_fraction(s, SHORT_THR),
            "lag-1 corr": next_interval_corr(s),
        }
    return pd.DataFrame(rows).T.round(3)


# HE: שתי טבלאות: הקלטה של 5 שניות (כמו במטלה) והקלטה ארוכה של 200 שניות. ההבדל ביניהן מראה כמה 5 שניות רועשות.
table_short = stats_table(short, T_SHORT)
table_long = stats_table(long_, T_LONG)
print("5 s recording"); display(table_short)
print("200 s recording"); display(table_long)

In [ ]:
# HE: בדיקות סבירות על הטבלה הארוכה: הקצב דומה בשלושתם, ו-CV ו-LV עולים מרגולרי לאקראי לפרצי.
t = table_long
assert (abs(t["rate (Hz)"] - RATE) < 1.0).all()
assert t.loc["Regular", "CV"] < t.loc["Irregular (random)", "CV"] < t.loc["Bursting", "CV"]
assert t.loc["Regular", "LV"] < t.loc["Irregular (random)", "LV"] < t.loc["Bursting", "LV"]
print("table checks passed")

**What step 4 shows (on Part 1's models)**

- **Comparison figure:** on the same axes the three distributions barely overlap. The regular neuron is a narrow peak near 100 ms. The random one is a wide hump around 100 ms. The bursting one has about three quarters of its intervals near 5 ms and the rest between 0.2 and 0.9 s. The y axis is the fraction of intervals in each bin (not `density=True`, which inflates the narrow bins of a log axis).
- **Table, 200 s:** the rates are all about 10 Hz (10.00, 9.99 and 10.27), yet CV (0.05, 0.98, 1.85), LV (0.003, 0.95, 1.42), Fano at 500 ms (0.02, 1.08, 1.46) and the short-interval fraction below 8 ms (0, 0.056, 0.749) separate the three neurons. The lag-1 correlation is about 0.02 for the regular and random neurons (no memory) and -0.26 for the bursting one: after a short interval a long one tends to come, and the other way round.
- **5 s versus 200 s:** in a 5 s recording the same neurons give rates between 9.0 and 10.2 Hz, a random-neuron Fano of 0.63 instead of 1.08, and a bursting Fano of 1.84 instead of 1.46, because 5 s holds only 10 windows of 500 ms. Numbers from one short recording are noisy; Q1 D measures that noise over many realizations.
- The short-interval threshold of 8 ms in the table was chosen from the threshold curve (step 3).

## Q1 B, step 5: how each neuron is built

Decision `b_figs` also asks for a diagram of how each neuron is built, so that the reader sees *why* the three neurons look different although they have the same mean rate of 10 Hz. Each row below is the recipe of one neuron in Part 1's simulation. The numbers are read from `q1_models.py`, so the diagram cannot drift away from the code.

In [ ]:
# HE: תרשים בנייה: שורה לכל נוירון, קופסאות וחצים. המספרים נשלפים מהקבועים ב-q1_models.py, כך שהתרשים לא ישתנה בלי הקוד.
from matplotlib.patches import FancyBboxPatch
from matplotlib.colors import to_rgba

mean_cycle, quiet_mean = q1_models.bursting_timing(RATE)
dead_ms = q1_models.DEAD_TIME * 1000
mean_ms = 1000 / RATE

# HE: בדיקה: המספרים בתרשים באמת נותנים 10Hz. בפרצי: ספייקים לפרץ חלקי אורך מחזור = קצב, ומחזור = פרץ + שקט.
assert np.isclose(q1_models.MEAN_SPIKES_PER_BURST / mean_cycle, RATE)
assert np.isclose((q1_models.MEAN_SPIKES_PER_BURST - 1) * q1_models.INTRA_MEAN + quiet_mean, mean_cycle)

refractory = f"never closer than\n{dead_ms:.0f} ms (refractory period)"
add_up = "add up the intervals\n= spike times"
recipes = [
    ("Regular", COLORS[0], [
        f"interval of about {mean_ms:.0f} ms,\nalmost no randomness\n(Gamma, shape k = {q1_models.K_REGULAR})",
        refractory, add_up]),
    ("Irregular (random)", COLORS[1], [
        f"interval of random length,\nmean {mean_ms:.0f} ms\n(Gamma, shape k = {q1_models.K_IRREGULAR},\nan exponential)",
        refractory, add_up]),
    ("Bursting", COLORS[2], [
        f"a burst of {q1_models.MIN_SPIKES} + Poisson({q1_models.EXTRA_SPIKES_MEAN}) spikes\n"
        f"(mean {q1_models.MEAN_SPIKES_PER_BURST}), about\n{q1_models.INTRA_MEAN * 1000:.0f} ms apart",
        f"quiet gap of at least\n{q1_models.QUIET_MIN * 1000:.0f} ms (mean {quiet_mean * 1000:.0f} ms)",
        "repeat until the recording\nis full, add up the times"]),
]

fig, ax = plt.subplots(figsize=(11, 4.4))
ax.set_xlim(0, 100); ax.set_ylim(0, 3); ax.axis("off")
xs, w, h = [19, 49, 78], 20, 0.8
for row, (name, color, texts) in enumerate(recipes):
    y = 2.5 - row
    ax.text(0, y, name, va="center", fontsize=10, fontweight="bold", color=color)
    for x, text in zip(xs, texts):
        ax.add_patch(FancyBboxPatch((x, y - h / 2), w, h, boxstyle="round,pad=0.02", fc=to_rgba(color, 0.15), ec=color))
        ax.text(x + w / 2, y, text, ha="center", va="center", fontsize=8.5)
    for x in xs[:2]:
        ax.annotate("", xy=(x + w + 9, y), xytext=(x + w, y), arrowprops=dict(arrowstyle="->"))
ax.set_title("How each neuron is built (the recipe in Part 1's simulation)")
plt.tight_layout(); save_fig(fig, "q1b_construction_diagram"); plt.show()

**What step 5 shows (on Part 1's models)**

- **Regular and irregular differ in one number.** Both draw an interval of mean 100 ms and never closer than 2 ms. The regular neuron uses a Gamma shape of 400 (the intervals are almost all the same) and the irregular neuron uses 1 (an exponential, as in a Poisson process, so the intervals are very variable).
- **The bursting neuron has two kinds of intervals:** about 5 ms inside a burst, and a quiet gap of at least 50 ms (mean 385 ms) between bursts. The number of spikes in a burst is random (mean 4).
- **All three have the same mean rate** because the numbers were chosen for it: a mean interval of 100 ms, or a burst cycle of 400 ms holding 4 spikes on average. The cell checks this with an `assert`.
- **What is deterministic and what is random:** the recipe and its numbers are fixed. Every interval, and for the bursting neuron every burst size, is a random draw. Using the same seed gives the same spike trains.

## Q1 D, step 1: new realizations, how much they vary, and which properties stay stable

The assignment asks to generate more realizations of the stochastic models, to see how much spike patterns vary between realizations, whether two realizations of the same neuron can look quite different, and which properties stay relatively stable.

Decisions applied: `d_n` (5 realizations shown as a raster, 1,000 for the statistics), `d_props` (spike count, CV, LV, short-interval fraction) and `d_plot` (box plot with all points, per neuron and per measure). Each realization is 5 s long, as in the assignment. On Part 1's models.

In [ ]:
# HE: 5 מימושים לכל נוירון, כל אחד עם זרע אחר. כל מימוש הוא "הקלטה" חדשה של אותו מודל.
N_SHOW = 5
realizations = [make_trains(T_SHORT, seed=100 + k) for k in range(N_SHOW)]   # HE: רשימה של 5, בכל אחת שלושה נוירונים

fig, axes = plt.subplots(3, 1, figsize=(10, 5.5), sharex=True)
for j, (ax, name, color) in enumerate(zip(axes, NAMES, COLORS)):
    trains = [realizations[k][j] for k in range(N_SHOW)]
    ax.eventplot(trains, colors=color, linelengths=0.7)
    ax.set_yticks(range(N_SHOW)); ax.set_yticklabels([f"run {k + 1} ({len(t)} spikes)" for k, t in enumerate(trains)], fontsize=8)
    ax.set_title(name, loc="left", fontsize=10)
axes[-1].set_xlabel("Time (s)"); axes[-1].set_xlim(0, T_SHORT)
fig.suptitle("Five realizations of each neuron (5 s each)")
plt.tight_layout(); save_fig(fig, "q1d_five_realizations"); plt.show()

In [ ]:
# HE: 1,000 מימושים לכל נוירון (החלטה d_n). לכל מימוש מחשבים את ארבעת המדדים של d_props.
N_STATS = 1000
PROPS = ["spike count", "CV", "LV", f"short ISI < {int(SHORT_THR * 1000)} ms"]
records = []
for k in range(N_STATS):
    trains = make_trains(T_SHORT, seed=10_000 + k)
    for name, s in zip(NAMES, trains):
        records.append({"neuron": name, "run": k, "spike count": len(s), "CV": cv(s), "LV": lv(s),
                        f"short ISI < {int(SHORT_THR * 1000)} ms": short_isi_fraction(s, SHORT_THR)})
runs = pd.DataFrame(records)

# HE: בדיקות: בדיוק 1,000 מימושים לכל נוירון, ואין nan (בהקלטה של 5 שניות בקצב 10Hz תמיד יש מספיק פעימות).
assert (runs.groupby("neuron").size() == N_STATS).all()
assert runs[PROPS].notna().all().all()
print(runs.groupby("neuron")[PROPS].median().round(3))

In [ ]:
# HE: גרף קופסה עם כל הנקודות (החלטה d_plot): לכל מדד ציור, ובו קופסה לכל נוירון ומעליה 1,000 הנקודות.
# HE: הקופסה מראה חציון ואת חצי האמצעי של המימושים; הנקודות מראות את כל הפיזור, כולל קיצוניים.
rng_jitter = np.random.default_rng(0)   # HE: רק להזזה אופקית קטנה של הנקודות כדי שלא יסתירו זו את זו
fig, axes = plt.subplots(1, 4, figsize=(13, 3.6))
for ax, prop in zip(axes, PROPS):
    data = [runs.loc[runs["neuron"] == n, prop].to_numpy() for n in NAMES]
    for i, (d, color) in enumerate(zip(data, COLORS)):
        ax.scatter(i + 1 + rng_jitter.uniform(-0.18, 0.18, len(d)), d, s=3, alpha=0.15, color=color)
    ax.boxplot(data, widths=0.5, showfliers=False)
    ax.set_xticks([1, 2, 3]); ax.set_xticklabels(["Reg", "Rand", "Burst"]); ax.set_title(prop)
fig.suptitle("1,000 realizations of 5 s: spread of each property")
plt.tight_layout(); save_fig(fig, "q1d_box_plots"); plt.show()

In [ ]:
# HE: כמה כל מדד יציב? לכל נוירון ומדד: חציון, טווח של 90% האמצעיים (אחוזון 5 עד 95),
# HE: והאם הטווחים של שלושת הנוירונים חופפים (אם לא חופפים, מימוש אחד מספיק כדי להבחין).
summary = {}
for prop in PROPS:
    for n in NAMES:
        v = runs.loc[runs["neuron"] == n, prop]
        summary[(prop, n)] = {"median": v.median(), "5th pct": v.quantile(0.05), "95th pct": v.quantile(0.95)}
summary = pd.DataFrame(summary).T.round(3)
display(summary)

for prop in PROPS:
    ranges = [(summary.loc[(prop, n), "5th pct"], summary.loc[(prop, n), "95th pct"]) for n in NAMES]
    lo_hi = sorted(ranges)
    overlap = any(lo_hi[i][1] >= lo_hi[i + 1][0] for i in range(len(lo_hi) - 1))
    print(f"{prop:18s} 90% ranges of the three neurons {'overlap' if overlap else 'do not overlap'}")

**What Q1 D step 1 shows (on Part 1's models)**

- **Can two realizations of the same neuron look quite different?** Yes, for the random and the bursting neuron. In the five 5 s runs the random neuron fires between 44 and 60 spikes and the bursting neuron between 46 and 73, with bursts in different places each time. The regular neuron hardly changes (50 or 51 spikes, the same even spacing).
- **Spike count (rate) is the least stable property:** over 1,000 runs the middle 90% is 49 to 51 for the regular neuron, 39 to 61 for the random one and 38 to 64 for the bursting one. The ranges overlap, so one 5 s recording cannot tell the neurons apart by rate. The bursting counts vary because each burst has a different number of spikes.
- **Shape properties are stable:** for CV, LV and the short-interval fraction the middle-90% ranges of the three neurons do not overlap (for example CV 0.04 to 0.06, 0.78 to 1.19 and 1.69 to 2.12). A single 5 s recording is usually enough to place a neuron by these measures. Q1 D step 2 tests this on new runs.

## Q1 D, step 2: telling the three patterns apart from observations alone

The assignment ends with: how would you tell the three firing patterns apart from observations, without knowing how they were generated? Decision `d_cls`: a two-dimensional view (CV against the short-interval fraction) and a simple threshold rule, evaluated on **new** simulations that were not used to set the thresholds. Decision `d_dur`: how the accuracy of that rule depends on the recording length (1, 2, 5, 10 and 30 s), in one figure.

The thresholds are set on the 1,000 runs of step 1 (the "training" runs). They are then frozen and tested on new runs with different seeds, so the accuracy is not measured on the same data that chose the thresholds.

In [ ]:
# HE: קובעים שני ספים מתוך ריצות האימון (1,000 ההרצות של שלב 1), כל סף באמצע הרווח בין שני נוירונים:
# HE: סף CV: באמצע בין אחוזון 95 של הרגולרי לאחוזון 5 של האקראי.
# HE: סף שיעור קצרים: באמצע בין אחוזון 95 של האקראי לאחוזון 5 של הפרצי.
SHORT_COL = f"short ISI < {int(SHORT_THR * 1000)} ms"
q = lambda n, col, a: runs.loc[runs["neuron"] == n, col].quantile(a)
CV_THR = (q("Regular", "CV", 0.95) + q("Irregular (random)", "CV", 0.05)) / 2
BURST_THR = (q("Irregular (random)", SHORT_COL, 0.95) + q("Bursting", SHORT_COL, 0.05)) / 2
print(f"CV threshold: {CV_THR:.3f}   short-interval-fraction threshold: {BURST_THR:.3f}")


def classify(cv_value, short_fraction):
    # HE: הכלל: קודם שואלים "יש הרבה מרווחים קצרים?" => פרצי. אחרת "המרווחים כמעט קבועים?" => רגולרי. אחרת => אקראי.
    # HE: אם אי אפשר לחשב את המדדים (מעט מדי פעימות) מחזירים "undecided" וזה נספר כטעות.
    if np.isnan(cv_value) or np.isnan(short_fraction):
        return "undecided"
    if short_fraction > BURST_THR:
        return "Bursting"
    if cv_value < CV_THR:
        return "Regular"
    return "Irregular (random)"


# HE: בדיקות קטנות לכלל עצמו, על ערכים שהתשובה שלהם ברורה.
assert classify(0.1, 0.0) == "Regular" and classify(1.0, 0.05) == "Irregular (random)"
assert classify(3.0, 0.8) == "Bursting" and classify(np.nan, 0.1) == "undecided"

In [ ]:
# HE: גרף דו-ממדי: כל נקודה היא הרצה אחת (מריצות האימון). הקווים המקווקווים הם שני הספים, שמחלקים את המישור לשלושה אזורים.
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for n, color in zip(NAMES, COLORS):
    d = runs[runs["neuron"] == n]
    ax.scatter(d["CV"], d[SHORT_COL], s=5, alpha=0.3, color=color, label=n)
ax.axvline(CV_THR, color="gray", ls="--", lw=1)
ax.axhline(BURST_THR, color="gray", ls="--", lw=1)
ax.set_xlabel("CV of the intervals"); ax.set_ylabel(f"Fraction of intervals < {int(SHORT_THR * 1000)} ms")
ax.legend(markerscale=3); ax.set_title("Each 5 s run as one point, with the two thresholds")
plt.tight_layout(); save_fig(fig, "q1d_rule_scatter"); plt.show()

In [ ]:
# HE: בדיקה על סימולציות חדשות: 1,000 הרצות חדשות לכל נוירון, עם זרעים שלא שימשו לקביעת הספים.
def evaluate(T, n_runs, seed0):
    rows = []
    for k in range(n_runs):
        for true, s in zip(NAMES, make_trains(T, seed=seed0 + k)):
            rows.append({"true": true, "predicted": classify(cv(s), short_isi_fraction(s, SHORT_THR))})
    return pd.DataFrame(rows)


test5 = evaluate(T_SHORT, 1000, seed0=50_000)
# HE: מטריצת בלבול: שורה = הנוירון האמיתי, עמודה = מה שהכלל אמר. האלכסון = תשובות נכונות.
confusion = pd.crosstab(test5["true"], test5["predicted"])
display(confusion)
acc5 = (test5["true"] == test5["predicted"]).mean()
print(f"accuracy on 3,000 new 5 s runs: {acc5:.3f}")
assert set(test5["true"].unique()) == set(NAMES) and len(test5) == 3000

In [ ]:
# HE: דיוק לפי אורך הקלטה (החלטה d_dur). הספים נשארים אלה שנקבעו על 5 שניות, וזה חלק מהשאלה:
# HE: האם כלל שנבנה על 5 שניות עובד גם על הקלטה קצרה או ארוכה יותר?
DURATIONS = [1, 2, 5, 10, 30]
acc = {}
for i, T in enumerate(DURATIONS):
    res = evaluate(float(T), 500, seed0=200_000 + 10_000 * i)
    correct = res["true"] == res["predicted"]
    acc[T] = correct.groupby(res["true"]).mean()
    acc[T]["all"] = correct.mean()
    acc[T]["undecided"] = (res["predicted"] == "undecided").mean()
acc = pd.DataFrame(acc).T
display(acc.round(3))

fig, ax = plt.subplots(figsize=(6.5, 3.8))
for n, color in zip(NAMES, COLORS):
    ax.plot(DURATIONS, acc[n], marker="o", color=color, label=n)
ax.plot(DURATIONS, acc["all"], marker="s", color="black", ls="--", label="all three")
ax.set_xscale("log"); ax.set_xticks(DURATIONS); ax.set_xticklabels(DURATIONS)
ax.set_xlabel("Recording length (s, log axis)"); ax.set_ylabel("Fraction classified correctly")
ax.set_ylim(0, 1.02); ax.legend(loc="lower right"); ax.set_title("How long a recording does the rule need?")
plt.tight_layout(); save_fig(fig, "q1d_accuracy_by_length"); plt.show()

**What Q1 D step 2 shows (on Part 1's models)**

- **The rule:** first "are there many short intervals?" (fraction above 0.41 means bursting), otherwise "are the intervals nearly constant?" (CV below 0.42 means regular), otherwise random. Both thresholds sit in the middle of the gap between neighbouring neurons in the 1,000 training runs.
- **On 3,000 new 5 s runs** the rule is right every time (1,000 of 1,000 for each neuron). In the two-dimensional plot the three clouds are far apart.
- **By recording length:** at 1 s the accuracy is 0.992 overall (bursting 0.992, random 0.984, regular 1.0), and 0.3% of the 1 s runs have too few spikes to compute the measures (counted as errors). From 2 s on it is 1.0.
- **Caveat:** this is easy because Part 1's three models are very different from each other by design. Real neurons overlap much more, so an accuracy of 1.0 here does not mean the rule would be this good on real recordings.
- **How to tell the patterns apart from observations alone:** not by the rate, which overlaps between neurons in a single recording, but by the shape of the intervals: how variable they are (CV) and how many are very short (the short-interval fraction), measured on a recording of at least a few seconds.